In [ ]:
# Required imports
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from enum import Enum
from datetime import datetime
import json
import math

print("✅ Imports successful")

## 1. Why Do Agents Need Memory?

Without memory, an agent:
- Forgets previous conversations
- Cannot learn from past experiences
- Repeats mistakes
- Cannot maintain context

Memory enables:
- **Continuity**: Remember past interactions
- **Learning**: Improve from experience
- **Personalization**: Remember user preferences
- **Context**: Maintain relevant information

In [ ]:
class MemoryType(Enum):
    """Types of memory systems."""
    SHORT_TERM = "short_term"      # Recent, temporary
    LONG_TERM = "long_term"        # Persistent storage
    EPISODIC = "episodic"          # Specific experiences
    SEMANTIC = "semantic"          # Facts and knowledge
    PROCEDURAL = "procedural"      # How to do things
    WORKING = "working"            # Active processing


@dataclass
class MemoryEntry:
    """A single memory entry."""
    content: str                          # The actual memory content
    memory_type: MemoryType              # Type of memory
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5              # 0-1 importance score
    metadata: Dict[str, Any] = field(default_factory=dict)
    access_count: int = 0                # How often accessed
    last_accessed: Optional[datetime] = None
    embedding: Optional[List[float]] = None  # Vector representation
    
    def __post_init__(self):
        if self.last_accessed is None:
            self.last_accessed = self.timestamp
    
    def access(self) -> None:
        """Record an access to this memory."""
        self.access_count += 1
        self.last_accessed = datetime.now()
    
    def recency_score(self) -> float:
        """Calculate how recent this memory is (0-1)."""
        age = (datetime.now() - self.timestamp).total_seconds()
        # Decay over 24 hours
        decay_hours = 24
        return math.exp(-age / (decay_hours * 3600))
    
    def relevance_score(self, query: str) -> float:
        """Simple keyword-based relevance (in production, use embeddings)."""
        query_words = set(query.lower().split())
        content_words = set(self.content.lower().split())
        overlap = len(query_words & content_words)
        if not query_words:
            return 0
        return overlap / len(query_words)

print("✅ Memory types and MemoryEntry defined")

## 2. Base Memory Class

In [ ]:
class BaseMemory(ABC):
    """
    Abstract base class for memory systems.
    """
    
    @abstractmethod
    def add(self, content: str, **kwargs) -> str:
        """Add a memory. Returns memory ID."""
        pass
    
    @abstractmethod
    def get(self, memory_id: str) -> Optional[MemoryEntry]:
        """Retrieve a specific memory."""
        pass
    
    @abstractmethod
    def search(self, query: str, limit: int = 5) -> List[MemoryEntry]:
        """Search for relevant memories."""
        pass
    
    @abstractmethod
    def clear(self) -> None:
        """Clear all memories."""
        pass
    
    @abstractmethod
    def size(self) -> int:
        """Return number of memories stored."""
        pass

print("✅ BaseMemory defined")

## 3. Short-term Memory (Conversation Buffer)

Stores recent conversation turns with a fixed window:

In [ ]:
@dataclass
class Message:
    """A single message in a conversation."""
    role: str  # 'user', 'assistant', 'system'
    content: str
    timestamp: datetime = field(default_factory=datetime.now)
    metadata: Dict[str, Any] = field(default_factory=dict)


class ConversationBuffer(BaseMemory):
    """
    Short-term memory: stores recent conversation history.
    Uses a sliding window to limit context size.
    """
    
    def __init__(self, max_messages: int = 20):
        self.max_messages = max_messages
        self.messages: List[Message] = []
        self._id_counter = 0
        self._id_to_index: Dict[str, int] = {}
    
    def add(self, content: str, role: str = "user", **kwargs) -> str:
        """Add a message to the conversation."""
        msg = Message(role=role, content=content, metadata=kwargs)
        self.messages.append(msg)
        
        # Generate ID
        msg_id = f"msg_{self._id_counter}"
        self._id_to_index[msg_id] = len(self.messages) - 1
        self._id_counter += 1
        
        # Trim if exceeding max
        if len(self.messages) > self.max_messages:
            self.messages.pop(0)
            # Update indices
            self._id_to_index = {
                k: v - 1 for k, v in self._id_to_index.items() 
                if v > 0
            }
        
        return msg_id
    
    def add_user_message(self, content: str) -> str:
        """Convenience method for user messages."""
        return self.add(content, role="user")
    
    def add_assistant_message(self, content: str) -> str:
        """Convenience method for assistant messages."""
        return self.add(content, role="assistant")
    
    def get(self, memory_id: str) -> Optional[MemoryEntry]:
        """Get a message by ID."""
        idx = self._id_to_index.get(memory_id)
        if idx is not None and 0 <= idx < len(self.messages):
            msg = self.messages[idx]
            return MemoryEntry(
                content=f"{msg.role}: {msg.content}",
                memory_type=MemoryType.SHORT_TERM,
                timestamp=msg.timestamp,
                metadata=msg.metadata
            )
        return None
    
    def search(self, query: str, limit: int = 5) -> List[MemoryEntry]:
        """Search messages by content."""
        results = []
        query_lower = query.lower()
        
        for msg in self.messages:
            if query_lower in msg.content.lower():
                results.append(MemoryEntry(
                    content=f"{msg.role}: {msg.content}",
                    memory_type=MemoryType.SHORT_TERM,
                    timestamp=msg.timestamp
                ))
        
        return results[:limit]
    
    def get_history(self, last_n: Optional[int] = None) -> List[Message]:
        """Get conversation history."""
        if last_n:
            return self.messages[-last_n:]
        return self.messages.copy()
    
    def get_formatted_history(self, last_n: Optional[int] = None) -> str:
        """Get history as formatted string."""
        messages = self.get_history(last_n)
        return "\n".join(f"{m.role}: {m.content}" for m in messages)
    
    def clear(self) -> None:
        """Clear all messages."""
        self.messages.clear()
        self._id_to_index.clear()
    
    def size(self) -> int:
        return len(self.messages)

print("✅ ConversationBuffer defined")

In [ ]:
# Test conversation buffer

buffer = ConversationBuffer(max_messages=5)

# Simulate a conversation
buffer.add_user_message("Hello, can you help me with Python?")
buffer.add_assistant_message("Of course! What do you need help with?")
buffer.add_user_message("How do I read a file?")
buffer.add_assistant_message("You can use open('file.txt', 'r') to read a file.")
buffer.add_user_message("What about writing to a file?")
buffer.add_assistant_message("Use open('file.txt', 'w') for writing.")

print("📝 Conversation History:")
print("-" * 50)
print(buffer.get_formatted_history())
print(f"\nBuffer size: {buffer.size()} messages")

In [ ]:
# Test buffer overflow (max_messages=5)

buffer.add_user_message("How do I append to a file?")
buffer.add_assistant_message("Use 'a' mode: open('file.txt', 'a')")

print("📝 After adding more messages (oldest trimmed):")
print("-" * 50)
print(buffer.get_formatted_history())
print(f"\nBuffer size: {buffer.size()} messages")

## 4. Summary Buffer Memory

Summarizes old conversations to save context space:

In [ ]:
class SummaryBuffer(BaseMemory):
    """
    Memory that summarizes older conversations.
    Keeps recent messages + summary of older ones.
    """
    
    def __init__(self, max_messages: int = 10, summary_threshold: int = 6):
        self.max_messages = max_messages
        self.summary_threshold = summary_threshold
        self.messages: List[Message] = []
        self.summary: str = ""
        self._id_counter = 0
    
    def add(self, content: str, role: str = "user", **kwargs) -> str:
        """Add a message."""
        msg = Message(role=role, content=content, metadata=kwargs)
        self.messages.append(msg)
        
        msg_id = f"summary_msg_{self._id_counter}"
        self._id_counter += 1
        
        # Summarize if exceeding threshold
        if len(self.messages) > self.max_messages:
            self._summarize_old_messages()
        
        return msg_id
    
    def _summarize_old_messages(self) -> None:
        """Summarize old messages to reduce context."""
        # Take oldest messages to summarize
        to_summarize = self.messages[:self.summary_threshold]
        self.messages = self.messages[self.summary_threshold:]
        
        # Create summary (in production, use an LLM)
        topics = set()
        for msg in to_summarize:
            words = msg.content.lower().split()
            # Extract key words (simple heuristic)
            for word in words:
                if len(word) > 5:
                    topics.add(word)
        
        new_summary = f"Discussed: {', '.join(list(topics)[:5])}"
        
        if self.summary:
            self.summary = f"{self.summary}; {new_summary}"
        else:
            self.summary = new_summary
    
    def get(self, memory_id: str) -> Optional[MemoryEntry]:
        """Get memory by ID (simplified)."""
        return None  # Simplified for demo
    
    def search(self, query: str, limit: int = 5) -> List[MemoryEntry]:
        """Search in summary and messages."""
        results = []
        query_lower = query.lower()
        
        # Check summary
        if query_lower in self.summary.lower():
            results.append(MemoryEntry(
                content=f"Summary: {self.summary}",
                memory_type=MemoryType.SHORT_TERM,
                importance=0.7
            ))
        
        # Check recent messages
        for msg in self.messages:
            if query_lower in msg.content.lower():
                results.append(MemoryEntry(
                    content=f"{msg.role}: {msg.content}",
                    memory_type=MemoryType.SHORT_TERM,
                    timestamp=msg.timestamp
                ))
        
        return results[:limit]
    
    def get_context(self) -> str:
        """Get full context including summary."""
        parts = []
        if self.summary:
            parts.append(f"[Previous conversation summary: {self.summary}]")
        parts.append("\nRecent messages:")
        for msg in self.messages:
            parts.append(f"{msg.role}: {msg.content}")
        return "\n".join(parts)
    
    def clear(self) -> None:
        self.messages.clear()
        self.summary = ""
    
    def size(self) -> int:
        return len(self.messages)

print("✅ SummaryBuffer defined")

In [ ]:
# Test summary buffer

summary_buffer = SummaryBuffer(max_messages=8, summary_threshold=4)

# Add many messages
conversations = [
    ("user", "Tell me about machine learning"),
    ("assistant", "Machine learning is a subset of AI that learns from data."),
    ("user", "What are neural networks?"),
    ("assistant", "Neural networks are computing systems inspired by biological neurons."),
    ("user", "How do I train a model?"),
    ("assistant", "Training involves feeding data and adjusting weights based on errors."),
    ("user", "What is gradient descent?"),
    ("assistant", "Gradient descent is an optimization algorithm to minimize loss."),
    ("user", "Can you explain backpropagation?"),
    ("assistant", "Backpropagation calculates gradients by propagating errors backwards."),
]

for role, content in conversations:
    summary_buffer.add(content, role=role)

print("📝 Context with Summary:")
print("=" * 60)
print(summary_buffer.get_context())

## 5. Long-term Memory (Semantic Store)

Persistent memory with similarity-based retrieval:

In [ ]:
class SemanticMemory(BaseMemory):
    """
    Long-term memory with semantic search.
    Uses simple TF-IDF-like similarity (in production, use embeddings).
    """
    
    def __init__(self):
        self.memories: Dict[str, MemoryEntry] = {}
        self._id_counter = 0
        self._term_doc_freq: Dict[str, int] = {}  # For IDF calculation
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization."""
        import re
        words = re.findall(r'\b\w+\b', text.lower())
        # Remove stopwords
        stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been',
                     'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                     'would', 'could', 'should', 'may', 'might', 'must', 'and',
                     'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with'}
        return [w for w in words if w not in stopwords and len(w) > 2]
    
    def _compute_tf(self, tokens: List[str]) -> Dict[str, float]:
        """Compute term frequency."""
        tf = {}
        for token in tokens:
            tf[token] = tf.get(token, 0) + 1
        # Normalize
        if tokens:
            max_freq = max(tf.values())
            tf = {k: v / max_freq for k, v in tf.items()}
        return tf
    
    def _similarity(self, query: str, memory: MemoryEntry) -> float:
        """Compute similarity between query and memory."""
        query_tokens = self._tokenize(query)
        memory_tokens = self._tokenize(memory.content)
        
        if not query_tokens or not memory_tokens:
            return 0.0
        
        query_tf = self._compute_tf(query_tokens)
        memory_tf = self._compute_tf(memory_tokens)
        
        # Simple dot product of TF vectors
        score = 0.0
        for term, tf in query_tf.items():
            if term in memory_tf:
                # Apply IDF-like weighting
                doc_freq = self._term_doc_freq.get(term, 1)
                idf = math.log(len(self.memories) / doc_freq + 1)
                score += tf * memory_tf[term] * idf
        
        return score
    
    def add(self, content: str, importance: float = 0.5, **kwargs) -> str:
        """Add a memory to long-term storage."""
        memory_id = f"mem_{self._id_counter}"
        self._id_counter += 1
        
        entry = MemoryEntry(
            content=content,
            memory_type=MemoryType.SEMANTIC,
            importance=importance,
            metadata=kwargs
        )
        
        self.memories[memory_id] = entry
        
        # Update term document frequencies
        tokens = set(self._tokenize(content))
        for token in tokens:
            self._term_doc_freq[token] = self._term_doc_freq.get(token, 0) + 1
        
        return memory_id
    
    def get(self, memory_id: str) -> Optional[MemoryEntry]:
        """Retrieve a specific memory."""
        entry = self.memories.get(memory_id)
        if entry:
            entry.access()
        return entry
    
    def search(self, query: str, limit: int = 5) -> List[MemoryEntry]:
        """Search for semantically similar memories."""
        if not self.memories:
            return []
        
        # Score all memories
        scored = []
        for memory in self.memories.values():
            sim = self._similarity(query, memory)
            # Combine with importance and recency
            score = sim * 0.6 + memory.importance * 0.3 + memory.recency_score() * 0.1
            scored.append((score, memory))
        
        # Sort by score
        scored.sort(key=lambda x: x[0], reverse=True)
        
        # Return top results
        results = [mem for score, mem in scored[:limit] if score > 0]
        for mem in results:
            mem.access()
        
        return results
    
    def clear(self) -> None:
        self.memories.clear()
        self._term_doc_freq.clear()
    
    def size(self) -> int:
        return len(self.memories)

print("✅ SemanticMemory defined")

In [ ]:
# Test semantic memory

semantic = SemanticMemory()

# Add various facts
facts = [
    ("Python was created by Guido van Rossum in 1991", 0.8),
    ("Machine learning models learn patterns from data", 0.9),
    ("Neural networks are inspired by biological neurons", 0.8),
    ("The capital of France is Paris", 0.7),
    ("Deep learning uses multiple neural network layers", 0.9),
    ("Python is popular for data science and AI", 0.8),
    ("GPT models use transformer architecture", 0.9),
    ("Coffee is a popular caffeinated beverage", 0.5),
]

print("Adding facts to semantic memory:")
for fact, importance in facts:
    mem_id = semantic.add(fact, importance=importance)
    print(f"  ✅ {mem_id}: {fact[:50]}...")

In [ ]:
# Search semantic memory

queries = [
    "Tell me about Python programming",
    "How do neural networks work?",
    "What is deep learning?"
]

print("🔍 Semantic Search Results:")
print("=" * 60)

for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 40)
    results = semantic.search(query, limit=3)
    for i, mem in enumerate(results, 1):
        print(f"  {i}. {mem.content}")

## 6. Episodic Memory

Stores specific experiences/events with temporal context:

In [ ]:
@dataclass
class Episode:
    """A specific experience or event."""
    event: str                # What happened
    context: str              # Where/when/why
    outcome: str              # Result/consequence
    emotions: List[str] = field(default_factory=list)  # Associated emotions
    actors: List[str] = field(default_factory=list)    # Who was involved
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5


class EpisodicMemory(BaseMemory):
    """
    Memory of specific experiences and events.
    Good for learning from past interactions.
    """
    
    def __init__(self, max_episodes: int = 100):
        self.max_episodes = max_episodes
        self.episodes: Dict[str, Episode] = {}
        self._id_counter = 0
    
    def add_episode(
        self, 
        event: str, 
        context: str, 
        outcome: str,
        emotions: List[str] = None,
        actors: List[str] = None,
        importance: float = 0.5
    ) -> str:
        """Add an episode to memory."""
        episode_id = f"ep_{self._id_counter}"
        self._id_counter += 1
        
        episode = Episode(
            event=event,
            context=context,
            outcome=outcome,
            emotions=emotions or [],
            actors=actors or [],
            importance=importance
        )
        
        self.episodes[episode_id] = episode
        
        # Remove oldest if at capacity
        if len(self.episodes) > self.max_episodes:
            oldest_id = min(self.episodes.keys(), key=lambda k: self.episodes[k].timestamp)
            del self.episodes[oldest_id]
        
        return episode_id
    
    def add(self, content: str, **kwargs) -> str:
        """Add from BaseMemory interface."""
        return self.add_episode(
            event=content,
            context=kwargs.get('context', ''),
            outcome=kwargs.get('outcome', ''),
            emotions=kwargs.get('emotions', []),
            actors=kwargs.get('actors', []),
            importance=kwargs.get('importance', 0.5)
        )
    
    def get(self, memory_id: str) -> Optional[MemoryEntry]:
        """Get an episode as MemoryEntry."""
        episode = self.episodes.get(memory_id)
        if episode:
            return MemoryEntry(
                content=f"{episode.event} | {episode.context} -> {episode.outcome}",
                memory_type=MemoryType.EPISODIC,
                timestamp=episode.timestamp,
                importance=episode.importance,
                metadata={'actors': episode.actors, 'emotions': episode.emotions}
            )
        return None
    
    def search(self, query: str, limit: int = 5) -> List[MemoryEntry]:
        """Search episodes by content."""
        query_lower = query.lower()
        results = []
        
        for ep_id, episode in self.episodes.items():
            # Check all fields
            text = f"{episode.event} {episode.context} {episode.outcome}"
            if query_lower in text.lower():
                entry = self.get(ep_id)
                if entry:
                    results.append(entry)
        
        # Sort by importance and recency
        results.sort(key=lambda x: (x.importance, x.recency_score()), reverse=True)
        return results[:limit]
    
    def find_similar_situations(self, current_situation: str, limit: int = 3) -> List[Episode]:
        """Find past episodes similar to current situation."""
        query_words = set(current_situation.lower().split())
        scored = []
        
        for episode in self.episodes.values():
            ep_words = set(f"{episode.event} {episode.context}".lower().split())
            overlap = len(query_words & ep_words)
            if overlap > 0:
                score = overlap / len(query_words)
                scored.append((score, episode))
        
        scored.sort(key=lambda x: x[0], reverse=True)
        return [ep for _, ep in scored[:limit]]
    
    def clear(self) -> None:
        self.episodes.clear()
    
    def size(self) -> int:
        return len(self.episodes)

print("✅ EpisodicMemory defined")

In [ ]:
# Test episodic memory

episodic = EpisodicMemory()

# Add some past experiences
episodes = [
    {
        "event": "User asked about Python installation",
        "context": "New user, Windows system",
        "outcome": "Successfully guided through python.org download",
        "emotions": ["helpful", "satisfied"],
        "actors": ["user", "assistant"],
        "importance": 0.7
    },
    {
        "event": "User had import error with numpy",
        "context": "Python 3.9, virtual environment",
        "outcome": "Fixed with pip install numpy in correct venv",
        "emotions": ["frustrated_user", "resolved"],
        "actors": ["user", "assistant"],
        "importance": 0.8
    },
    {
        "event": "Explained machine learning basics",
        "context": "Student learning AI",
        "outcome": "User understood supervised vs unsupervised",
        "emotions": ["educational", "positive"],
        "actors": ["student", "assistant"],
        "importance": 0.6
    },
]

print("Adding episodes:")
for ep in episodes:
    ep_id = episodic.add_episode(**ep)
    print(f"  ✅ {ep_id}: {ep['event'][:40]}...")

In [ ]:
# Find similar situations

current = "User is having trouble with pip install"

print(f"\n🔍 Finding similar situations to: '{current}'")
print("-" * 50)

similar = episodic.find_similar_situations(current)

for i, ep in enumerate(similar, 1):
    print(f"\n{i}. Event: {ep.event}")
    print(f"   Context: {ep.context}")
    print(f"   Outcome: {ep.outcome}")
    print(f"   → Lesson: What worked before might work again!")

## 7. Unified Memory Manager

Combines all memory types into a coherent system:

In [ ]:
class MemoryManager:
    """
    Unified memory system combining all memory types.
    """
    
    def __init__(self):
        self.short_term = ConversationBuffer(max_messages=20)
        self.long_term = SemanticMemory()
        self.episodic = EpisodicMemory()
    
    def add_message(self, content: str, role: str = "user") -> str:
        """Add a conversation message."""
        return self.short_term.add(content, role=role)
    
    def add_fact(self, content: str, importance: float = 0.5) -> str:
        """Add a fact to long-term memory."""
        return self.long_term.add(content, importance=importance)
    
    def add_experience(
        self, 
        event: str, 
        context: str, 
        outcome: str,
        **kwargs
    ) -> str:
        """Add an experience to episodic memory."""
        return self.episodic.add_episode(event, context, outcome, **kwargs)
    
    def get_context(self, query: str = "", max_items: int = 10) -> str:
        """
        Get relevant context from all memory types.
        This is what gets sent to the LLM.
        """
        parts = []
        
        # Recent conversation
        recent = self.short_term.get_formatted_history(last_n=5)
        if recent:
            parts.append("=== Recent Conversation ===")
            parts.append(recent)
        
        # Relevant facts
        if query:
            facts = self.long_term.search(query, limit=3)
            if facts:
                parts.append("\n=== Relevant Facts ===")
                for fact in facts:
                    parts.append(f"• {fact.content}")
        
        # Similar past experiences
        if query:
            experiences = self.episodic.find_similar_situations(query, limit=2)
            if experiences:
                parts.append("\n=== Past Experiences ===")
                for exp in experiences:
                    parts.append(f"• {exp.event} → {exp.outcome}")
        
        return "\n".join(parts)
    
    def reflect_and_consolidate(self) -> List[str]:
        """
        Reflect on recent conversations and consolidate to long-term memory.
        In production, this would use an LLM to extract key insights.
        """
        consolidated = []
        
        # Get recent conversation
        history = self.short_term.get_history()
        
        # Simple extraction: look for definitions and explanations
        for msg in history:
            if msg.role == "assistant" and len(msg.content) > 50:
                # Check if it looks like a fact/definition
                if " is " in msg.content or " are " in msg.content:
                    fact = msg.content[:200]  # Truncate
                    mem_id = self.long_term.add(fact, importance=0.6)
                    consolidated.append(f"Consolidated: {fact[:50]}...")
        
        return consolidated
    
    def get_stats(self) -> Dict[str, int]:
        """Get memory statistics."""
        return {
            "short_term_messages": self.short_term.size(),
            "long_term_facts": self.long_term.size(),
            "episodic_experiences": self.episodic.size()
        }

print("✅ MemoryManager defined")

In [ ]:
# Test the unified memory manager

memory = MemoryManager()

# Add some facts
memory.add_fact("Python is a programming language created in 1991", importance=0.8)
memory.add_fact("Machine learning is a subset of artificial intelligence", importance=0.9)
memory.add_fact("Transformers are neural network architectures for NLP", importance=0.9)

# Add an experience
memory.add_experience(
    event="Helped user debug Python code",
    context="IndentationError in for loop",
    outcome="Fixed by correcting whitespace",
    importance=0.7
)

# Add conversation
memory.add_message("Can you help me with machine learning?", role="user")
memory.add_message("Of course! Machine learning is a field where computers learn patterns from data.", role="assistant")
memory.add_message("What Python libraries should I use?", role="user")

print("📊 Memory Stats:")
print(memory.get_stats())

In [ ]:
# Get full context for LLM

query = "Python machine learning libraries"

print(f"\n🧠 Full Context for query: '{query}'")
print("=" * 60)
print(memory.get_context(query))

## Summary

We built a complete memory system with:

| Memory Type | Purpose | Implementation |
|------------|---------|----------------|
| **Short-term** | Recent conversation | `ConversationBuffer` |
| **Summary** | Compressed history | `SummaryBuffer` |
| **Semantic** | Facts & knowledge | `SemanticMemory` |
| **Episodic** | Past experiences | `EpisodicMemory` |
| **Unified** | Combined system | `MemoryManager` |

### Key Concepts:
- **Recency**: Recent memories are more relevant
- **Importance**: Not all memories are equal
- **Similarity Search**: Find relevant memories
- **Consolidation**: Move short-term to long-term
- **Context Building**: Combine memories for LLM